Code based on Svet's notebook

In [17]:
import glob
import cudf
import pandas as pd
import tqdm as tqdm
import os

os.chdir('/home/rapids/Capstone')


In [18]:
# Function to convert a CSV file to a Parquet file
def convert_csv_to_parquet(file_path):
    # Read the CSV file
    df = cudf.read_csv(file_path)

    # Construct the new file path for the Parquet file
    parquet_file_path = file_path.replace('.csv', '.parquet')

    # Write the DataFrame to a Parquet file
    df.to_parquet(parquet_file_path, index=False)

    # Delete the DataFrame to free up memory
    del df

# Get the list of CSV files
file_list = glob.glob('data/*.csv')
print(f"Found files: {file_list}")

# Process files sequentially
for file_path in file_list:
    convert_csv_to_parquet(file_path)

    
    

Found files: ['data/2023_01_Gener_BicingNou_ESTACIONS.csv', 'data/2021_10_Octubre_BicingNou_ESTACIONS.csv', 'data/2024_04_Abril_BicingNou_ESTACIONS.csv', 'data/2024_03_Marc_BicingNou_ESTACIONS.csv', 'data/2022_01_Gener_BicingNou_ESTACIONS.csv', 'data/2023_11_Novembre_BicingNou_ESTACIONS.csv', 'data/2022_12_Desembre_BicingNou_ESTACIONS.csv', 'data/2024_02_Febrer_BicingNou_ESTACIONS.csv', 'data/2022_05_Maig_BicingNou_ESTACIONS.csv', 'data/2022_10_Octubre_BicingNou_ESTACIONS.csv', 'data/2024_05_Maig_BicingNou_ESTACIONS.csv', 'data/2020_01_Gener_BicingNou_ESTACIONS.csv', 'data/2022_09_Setembre_BicingNou_ESTACIONS.csv', 'data/2023_07_Juliol_BicingNou_ESTACIONS.csv', 'data/2023_08_Agost_BicingNou_ESTACIONS.csv', 'data/2021_11_Novembre_BicingNou_ESTACIONS.csv', 'data/2021_12_Desembre_BicingNou_ESTACIONS.csv', 'data/2022_04_Abril_BicingNou_ESTACIONS.csv', 'data/2020_08_Agost_BicingNou_ESTACIONS.csv', 'data/2023_02_Febrer_BicingNou_ESTACIONS.csv', 'data/2020_05_Maig_BicingNou_ESTACIONS.csv', 'd

In [19]:
# For each parquet file, read it and, if present, delete the columns 'traffic', 'V1' and 'last_updated'
for file_path in glob.glob('data/*.parquet'):
    # Read the Parquet file
    df = cudf.read_parquet(file_path)

    # Drop the columns if they exist
    columns_to_drop = ['traffic', 'V1', 'last_updated']
    for column in columns_to_drop:
        if column in df.columns:
            df.drop(column, axis=1, inplace=True)

    # Write the DataFrame back to the Parquet file
    df.to_parquet(file_path, index=False)

    # Delete the DataFrame to free up memory
    del df

In [20]:
# For every parquet file, replace any negative value in the numeric columns with zero
for file_path in glob.glob('data/*.parquet'):
    # Read the Parquet file
    df = cudf.read_parquet(file_path)

    # Replace negative values with zero
    for column in df.columns:
        if df[column].dtype in ['int8', 'int16', 'int32', 'int64', 'float32', 'float64']:
            df[column] = df[column].clip(lower=0)

    # Write the DataFrame back to the Parquet file
    df.to_parquet(file_path, index=False)

    # Delete the DataFrame to free up memory
    del df

In [21]:
# Convert the 'last_reported' column from Unix timestamp to datetime. After that, sort the dataframe in ascending order. For each day and hour, keep average of the numerical columns and the mode of the categorical ones. Reset the index after everything is done.

# For every parquet file, replace any negative value in the numeric columns with zero
for file_path in glob.glob('data/*.parquet'):
    # Read the Parquet file
    df = cudf.read_parquet(file_path)

    # Sort the DataFrame in ascending order of 'last_reported'
    df = df.sort_values('last_reported')

    # Convert 'last_reported' column from Unix timestamp to datetime
    if 'last_reported' in df.columns:
        df['last_reported'] = cudf.to_datetime(df['last_reported'], unit='s')

    

    # Create two new columns for day and hour
    df['day'] = df['last_reported'].dt.strftime('%Y-%m-%d')  # Extract date as string
    df['hour'] = df['last_reported'].dt.hour

    # Reset the index
    df = df.reset_index(drop=True)

    # Write the DataFrame back to the Parquet file
    df.to_parquet(file_path, index=False)

    # Delete the DataFrame to free up memory
    del df



In [22]:
df = cudf.read_parquet('data/2023_01_Gener_BicingNou_ESTACIONS.parquet')
df.head(10)

,station_id,num_bikes_available,num_bikes_available_types.mechanical,num_bikes_available_types.ebike,num_docks_available,last_reported,is_charging_station,status,is_installed,is_renting,is_returning,ttl,day,hour
0,448,4,4,0,23,2022-12-31 22:55:23,True,IN_SERVICE,1,1,1,19,2022-12-31,22
1,198,2,1,1,25,2022-12-31 22:55:26,True,IN_SERVICE,1,1,1,19,2022-12-31,22
2,25,13,10,3,7,2022-12-31 22:55:28,True,IN_SERVICE,1,1,1,19,2022-12-31,22
3,463,10,0,10,14,2022-12-31 22:55:29,True,IN_SERVICE,1,1,1,19,2022-12-31,22
4,202,3,1,2,24,2022-12-31 22:55:30,True,IN_SERVICE,1,1,1,19,2022-12-31,22
5,281,13,0,13,11,2022-12-31 22:55:31,True,IN_SERVICE,1,1,1,19,2022-12-31,22
6,338,4,1,3,31,2022-12-31 22:55:31,True,IN_SERVICE,1,1,1,19,2022-12-31,22
7,268,18,14,4,9,2022-12-31 22:55:32,True,IN_SERVICE,1,1,1,19,2022-12-31,22
8,462,10,0,10,18,2022-12-31 22:55:32,True,IN_SERVICE,1,1,1,19,2022-12-31,22
9,47,21,21,0,26,2022-12-31 22:55:33,True,IN_SERVICE,1,1,1,19,2022-12-31,22


In [23]:
!git config --global user.email "github@botello.me"
!git config --global user.name "LEBsci"

/usr/bin/sh: 1: git: not found
/usr/bin/sh: 1: git: not found
